# 0-order optimization

In [ ]:
# pip install -e .
# python -m cell_simulator.search_example

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from cell_simulator.dynamics import HarmonicOscillatorContraction
from cell_simulator.forces import ForceModel
from cell_simulator.geometry import Cell, PillarGrid
from cell_simulator.integrators import RK4
from cell_simulator.observations import TrackingObservationModel
from cell_simulator.simulator import Experiment, ExperimentConfig
from cell_simulator.visualize import *

### Setup physics

In [ ]:
set_style()
grid = PillarGrid.square_lattice(15, 15, 4000.0)
cell = Cell(center_nm=np.array([28900.0, 27400.0]), radius_nm=20000.0)
force_model = ForceModel(F_max=600.0, k=4.8)
times = np.arange(0.0, 30.0 + 1e-9, 0.5)

### Setup simulation + loss

In [ ]:
def simulate(omega, phase, observation_model=None):
    cfg = ExperimentConfig(
        pillar_grid=grid,
        cell=cell,
        dynamics=HarmonicOscillatorContraction(omega, phase),
        integrator=RK4(),
        force_model=force_model,
        observation_model=observation_model,
    )
    return Experiment(cfg).run(times, steps_per_frame=10)


def loss(result):
    return float(np.mean((result.forces - target) ** 2))

In [ ]:
target.shape

In [ ]:
grid.n_pillars

### Define GT parameters

In [ ]:
true_omega, true_phase = 2 * np.pi / 10.0, 0.8
true_data = simulate(
    true_omega, true_phase, observation_model=TrackingObservationModel(sigma_nm=0.0, rng=np.random.default_rng(1))
)
target = true_data.observed_forces

noise_floor = loss(true_data)  # true parameters, noise-free prediction vs noisy target
print(f"Noise floor: {noise_floor:.2f} i.e. minimum achievable loss with perfect parameters")

## Scalar fit: contraction signal $a(t)$

As a first step we fit the scalar contraction signal $a(t) = \tfrac{1}{2}\big(1 + \sin(\omega t + \varphi)\big)$ directly, before fitting the force field.
$a(t)$ is not observable in real TFM data.

In [ ]:
def contraction(omega, phase):
    # Contraction signal a(t) on the same time grid as the simulator (matches panel (a) of plot_fit_comparison)
    return 0.5 * (1 + np.sin(omega * times + phase))


sigma_a = 2.0 * 4.8 / 600.0  # tracking noise x stiffness / F_max -> same relative noise as the forces
a_true = contraction(true_omega, true_phase)
target_a = a_true  # + np.random.default_rng(2).normal(0.0, sigma_a, a_true.shape)


def loss_a(omega, phase):
    return float(np.mean((contraction(omega, phase) - target_a) ** 2))


noise_floor_a = loss_a(true_omega, true_phase)
print(f"Noise floor (a): {noise_floor_a:.2e}  (sigma_a^2 = {sigma_a**2:.2e}, should roughly agree)")

### Random search on $a(t)$

Same $N$, priors and seed as the force-field search below, so the candidates are identical.

In [ ]:
N = 1000
rng = np.random.default_rng(0)  # same seed as the force-field search -> same candidates

omegas_a = np.abs(rng.normal(0.6, 0.15, N))
phis_a = rng.normal(0.0, 1.5, N)

losses_a = np.array([loss_a(w, p) for w, p in zip(omegas_a, phis_a)])
best_a, worst_a = int(np.argmin(losses_a)), int(np.argmax(losses_a))

print(f"Best:  omega={omegas_a[best_a]:.3f}, phi={phis_a[best_a]:+.2f}, loss={losses_a[best_a]:.2e}")
print(f"Worst: omega={omegas_a[worst_a]:.3f}, phi={phis_a[worst_a]:+.2f}, loss={losses_a[worst_a]:.2e}")

fig = plot_parameter_search(omegas_a, phis_a, losses_a, true_omega, true_phase, noise_floor_a)

### Loss landscape on $a(t)$ (fast, no force simulation needed)

In [ ]:
omega_grid_a = np.linspace(0.01, 1.1, 200)
phi_grid_a = np.linspace(-np.pi, np.pi, 200)
loss_grid_a = np.array([[loss_a(w, p) for w in omega_grid_a] for p in phi_grid_a])

fig, ax = plt.subplots(figsize=(6.5, 4.8))
fig = plot_loss_landscape(
    omega_grid_a, phi_grid_a, loss_grid_a, true_omega, true_phase, samples=(omegas_a, phis_a), ax=ax
)

### Compare worst, best and true fit on $a(t)$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(times, target_a, ".", color="grey", label="observed (target)")
ax.plot(times, a_true, color="k", label=f"truth: ω={true_omega:.3f}, φ={true_phase:+.2f}")
ax.plot(
    times,
    contraction(omegas_a[best_a], phis_a[best_a]),
    label=f"best: ω={omegas_a[best_a]:.3f}, φ={phis_a[best_a]:+.2f}",
)
ax.plot(
    times,
    contraction(omegas_a[worst_a], phis_a[worst_a]),
    color="tab:red",
    label=f"worst: ω={omegas_a[worst_a]:.3f}, φ={phis_a[worst_a]:+.2f}",
)
ax.set_xlabel("time [s]")
ax.set_ylabel("a(t)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## Force-field fit

### Simulate random parameters

In [ ]:
N = 1000
rng = np.random.default_rng(0)

# Sample random parameters
omegas = np.abs(rng.normal(0.6, 0.15, N))
phis = rng.normal(0.0, 1.5, N)

# Simulate
candidates = [simulate(w, p) for w, p in zip(omegas, phis)]

# Score candidates
losses = np.array([loss(r) for r in candidates])
best, worst = candidates[int(np.argmin(losses))], candidates[int(np.argmax(losses))]

In [ ]:
fig = plot_parameter_search(omegas, phis, losses, true_omega, true_phase, noise_floor)

(a) The yellow region runs diagonally: a slightly higher ω pairs with a slightly lower φ, and vice versa. The two parameters partly compensate for each other. A too fast frequency's loss can be offset by a shifted phase starting behind such that it does not "run off" too much within our interval. 

(b) Largest loss is at true $\omega$ (interesting) but at the opposite phase, shifted such that it is always opposite. Away from the true $\omega$ the loss plateaus around $8\cdot 10^3$ as the frequency is very wrong and the phase is less important. 

(c) Many true $\phi$ have high loss because their $\omega$ is so wrong. 

The plots also underlines that random sampling is a wasteful tactic. Only a handful og samples lands inside the area of interest (a) i.e. funnels (b, c) and only by luck. 

### As grid search (takes 3 min)

In [ ]:
omega_grid = np.linspace(0.01, 1.1, 100)
phi_grid = np.linspace(-np.pi, np.pi, 100)
loss_grid = np.array([[loss(simulate(w, p)) for w in omega_grid] for p in phi_grid])
fig, ax = plt.subplots(figsize=(6.5, 4.8))
fig = plot_loss_landscape(omega_grid, phi_grid, loss_grid, true_omega, true_phase, samples=(omegas, phis), ax=ax)

### Compare worst, best and true fit

In [ ]:
edge_pillar = int(np.argmax(true_data.rho * true_data.under_cell))  # strongest-force pillar
fig = plot_fit_comparison(true_data, best, worst, pillar_idx=edge_pillar)

Again notice, true $\omega$ opposite $\phi$ is the worst fit. Also notice that the best fit accumulate the loss as time progresses as the frequency is a bit (0.002) slower than the true value. 

In [ ]:
from IPython.display import HTML

anim = animate_fit_comparison(true_data, best, worst, fps=5)
plt.close(anim._fig)  # avoid an extra static copy below the video
HTML(anim.to_jshtml())

In [ ]:
from IPython.display import HTML

anim = animate_pillar_motion(true_data, edge_pillar, fps=5)
plt.close(anim._fig)  # avoid an extra static copy below the video
HTML(anim.to_jshtml())

Different values for $u_x$ and $u_y$ because the pillar (175) is not placed with the same distance from the cell center so the force vector is different lengths in the two dimensions.